In [ ]:
!pip install torch
!pip install torchvision
!pip install tqdm

import torch
import torch.nn as nn
from torchvision import datasets, transforms
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

BATCH_SIZE = 64

transform = transforms.Compose({
    transforms.ToTensor(),
})

train_dataset = datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

train_loader = torch.utils.data.DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = torch.utils.data.DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

class MNISTCNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(1, 32, 3, 1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.1),
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, 1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.1),
        )
        self.layer3 = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*5*5, 256),
            nn.ReLU(),
            nn.Linear(256, 10),
            nn.LogSoftmax(dim=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        return x

model = MNISTCNNModel().to(device)
criterion = nn.NLLLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(10):
    total_loss = 0
    for images, labels in tqdm(train_loader):
        optimizer.zero_grad()
        outputs = model(images.to(device))
        loss = criterion(outputs, labels.to(device))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Epoch: {epoch+1}, Loss: {total_loss/len(train_loader)}')

correct = 0
total = 0
model.eval()
with torch.no_grad():
    for images, labels in tqdm(test_loader):
        outputs = model(images.to(device))
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels.to(device)).sum().item()
print(f"Accuracy: {100 * correct/total}%")

100%|██████████| 938/938 [00:10<00:00, 89.57it/s] 


Epoch: 1, Loss: 0.1751138279527259


100%|██████████| 938/938 [00:09<00:00, 94.24it/s] 


Epoch: 2, Loss: 0.054477872723129706


100%|██████████| 938/938 [00:09<00:00, 98.68it/s] 


...

Epoch: 9, Loss: 0.011485951234742404


100%|██████████| 938/938 [00:09<00:00, 101.86it/s]


Epoch: 10, Loss: 0.00961071034555298


100%|██████████| 157/157 [00:01<00:00, 98.60it/s] 

Accuracy: 99.18%
